# Taller evaluable: SQL Server y T-SQL
## Módulo 2 · Bloque de taller · Caso: biblioteca

Diplomado en Estrategias de Datos: Python, SQL Server y Big Data. Facultad de Estadística, Universidad Santo Tomás.

Este cuaderno se resuelve desde Python, conectado al motor SQL Server del Codespace. Antes de comenzar, el entorno de Python debe estar preparado y el kernel seleccionado, conforme al manual de consultas desde Python. Complete cada celda donde se indica.

## Parte práctica

### 0. Conexión y creación de la base de datos
Las dos celdas siguientes se conectan al motor, crean la base `biblioteca` y se conectan a ella. Ejecútelas tal cual.

In [ ]:
import pymssql
import pandas as pd
import warnings
warnings.filterwarnings("ignore")   # oculta avisos no criticos de pandas

# Conexion a 'master' para crear la base. Si ya existe, se fuerza el cierre
# de otras conexiones antes de borrarla, para evitar el error 'base en uso'.
con_master = pymssql.connect(server="db", user="sa",
                             password="TuClave_Fuerte123",
                             database="master", autocommit=True)
cur = con_master.cursor()
cur.execute("""
IF DB_ID('biblioteca') IS NOT NULL
BEGIN
    ALTER DATABASE biblioteca SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE biblioteca;
END
""")
cur.execute("CREATE DATABASE biblioteca")
con_master.close()
print("Base de datos biblioteca creada")

In [ ]:
# Conexion a la base biblioteca, que se usara en todo el taller
con = pymssql.connect(server="db", user="sa",
                      password="TuClave_Fuerte123",
                      database="biblioteca")
print("Conectado a biblioteca")

### 1. Crear las tablas
Cree las tablas `lectores` y `prestamos` con esta especificación:

**lectores**: `lector_id` (entero, clave primaria), `nombre` (texto, obligatorio), `programa` (texto, admite nulo), `sede` (texto, obligatorio), `fecha_inscripcion` (fecha).

**prestamos**: `prestamo_id` (entero, clave primaria), `lector_id` (entero, clave foránea a `lectores`), `categoria` (texto), `titulo` (texto), `dias_prestamo` (entero), `multa` (decimal de 10 dígitos con 2 decimales), `fecha_prestamo` (fecha).

In [ ]:
cur = con.cursor()

# Escriba aqui el CREATE TABLE de lectores y el de prestamos,
# segun la especificacion del enunciado. Use cur.execute(""" ... """).
# Recuerde: lectores tiene clave primaria lector_id;
# prestamos tiene clave primaria prestamo_id y clave foranea lector_id.

# cur.execute(""" CREATE TABLE lectores ( ... ) """)
# cur.execute(""" CREATE TABLE prestamos ( ... ) """)

con.commit()
print("Tablas creadas")

### 2. Cargar los datos
Ejecute esta celda para cargar los datos de ejemplo (sintéticos e ilustrativos).

In [ ]:
# Datos sinteticos e ilustrativos (no son cifras reales)
lectores = [
    (1, 'Mariana Ospina', 'Medicina', 'Bogotá', '2023-02-19'),
    (2, 'Julián Cárdenas', 'Ingeniería', 'Chía', '2023-10-13'),
    (3, 'Natalia Bermúdez', 'Medicina', 'Chía', '2023-03-16'),
    (4, 'Sebastián Arango', None, 'Chía', '2025-07-28'),
    (5, 'Carolina Peña', 'Psicología', 'Chía', '2024-03-17'),
    (6, 'Esteban Gil', 'Economía', 'Villavicencio', '2023-07-17'),
    (7, 'Laura Montoya', None, 'Bogotá', '2023-02-10'),
    (8, 'Mateo Guerrero', 'Derecho', 'Chía', '2025-04-22'),
    (9, 'Isabela Cortés', 'Derecho', 'Chía', '2025-11-01'),
    (10, 'Samuel Rincón', 'Psicología', 'Chía', '2023-10-17'),
    (11, 'Gabriela León', 'Medicina', 'Bogotá', '2023-09-18'),
    (12, 'Nicolás Acosta', 'Psicología', 'Chía', '2023-10-03'),
    (13, 'Manuela Torres', 'Medicina', 'Tunja', '2024-10-27'),
    (14, 'David Salazar', 'Derecho', 'Chía', '2025-12-26'),
    (15, 'Antonia Vega', 'Ingeniería', 'Bogotá', '2025-08-04'),
    (16, 'Emilio Cárdenas', 'Medicina', 'Chía', '2023-12-25'),
    (17, 'Renata Ochoa', 'Derecho', 'Villavicencio', '2023-09-15'),
    (18, 'Simón Duarte', 'Psicología', 'Tunja', '2023-05-14'),
]

prestamos = [
    (1, 17, 'Tecnico', 'Calculo', 39, 0, '2026-06-21'),
    (2, 7, 'Ciencia', 'Cosmos', 11, 5400, '2026-06-15'),
    (3, 10, 'Historia', 'El Bogotazo', 5, 12500, '2026-05-15'),
    (4, 14, 'Novela', 'Delirio', 6, 3200, '2026-02-17'),
    (5, 7, 'Novela', 'Los ejercitos', 15, 1500, '2026-06-28'),
    (6, 7, 'Ciencia', 'El gen egoista', 41, 12500, '2026-06-25'),
    (7, 4, 'Tecnico', 'Bases de datos', 20, 0, '2026-01-23'),
    (8, 15, 'Tecnico', 'Algebra lineal', 6, 1500, '2026-05-13'),
    (9, 2, 'Tecnico', 'Calculo', 5, 0, '2026-04-17'),
    (10, 1, 'Novela', 'Los ejercitos', 24, 12500, '2026-01-11'),
    (11, 3, 'Novela', 'El olvido que seremos', 44, 1500, '2026-06-15'),
    (12, 8, 'Ciencia', 'Sapiens', 9, 21000, '2026-01-15'),
    (13, 15, 'Historia', 'El Bogotazo', 15, 0, '2026-01-26'),
    (14, 3, 'Historia', '1810', 23, 12500, '2026-01-14'),
    (15, 4, 'Historia', 'Bolivar', 28, 21000, '2026-03-22'),
    (16, 17, 'Novela', 'Los ejercitos', 9, 0, '2026-05-10'),
    (17, 6, 'Novela', 'La voragine', 44, 0, '2026-05-04'),
    (18, 4, 'Historia', 'El Bogotazo', 8, 8000, '2026-03-20'),
    (19, 7, 'Historia', 'Historia minima', 15, 5400, '2026-01-22'),
    (20, 1, 'Historia', 'Historia minima', 27, 0, '2026-04-07'),
    (21, 1, 'Ciencia', 'El gen egoista', 36, 5400, '2026-01-13'),
    (22, 17, 'Historia', 'La violencia en Colombia', 9, 21000, '2026-02-27'),
    (23, 10, 'Ciencia', 'El origen', 5, 21000, '2026-03-22'),
    (24, 3, 'Ciencia', 'Cosmos', 40, 3200, '2026-05-07'),
    (25, 5, 'Historia', 'La violencia en Colombia', 14, 0, '2026-01-13'),
    (26, 10, 'Ciencia', 'Sapiens', 6, 0, '2026-01-06'),
    (27, 8, 'Novela', 'Cien años de soledad', 3, 5400, '2026-04-16'),
    (28, 8, 'Historia', 'La violencia en Colombia', 28, 1500, '2026-02-07'),
    (29, 11, 'Ciencia', 'El gen egoista', 5, 0, '2026-06-13'),
    (30, 7, 'Tecnico', 'Algebra lineal', 19, 8000, '2026-03-22'),
    (31, 16, 'Tecnico', 'Calculo', 10, 8000, '2026-01-04'),
    (32, 10, 'Ciencia', 'El gen egoista', 23, 5400, '2026-03-22'),
    (33, 11, 'Novela', 'Cien años de soledad', 23, 0, '2026-01-19'),
    (34, 1, 'Ciencia', 'Sapiens', 18, 0, '2026-01-09'),
    (35, 18, 'Tecnico', 'Bases de datos', 13, 0, '2026-03-28'),
    (36, 14, 'Ciencia', 'Breve historia del tiempo', 44, 3200, '2026-02-13'),
    (37, 7, 'Historia', 'El Bogotazo', 36, 3200, '2026-06-05'),
    (38, 16, 'Ciencia', 'Breve historia del tiempo', 11, 0, '2026-03-06'),
    (39, 3, 'Tecnico', 'Calculo', 17, 0, '2026-01-12'),
    (40, 11, 'Novela', 'Los ejercitos', 5, 3200, '2026-01-06'),
    (41, 16, 'Historia', 'La violencia en Colombia', 42, 5400, '2026-06-25'),
    (42, 1, 'Novela', 'Cien años de soledad', 22, 21000, '2026-02-24'),
    (43, 17, 'Tecnico', 'Calculo', 17, 1500, '2026-06-11'),
    (44, 8, 'Tecnico', 'Redes', 23, 1500, '2026-03-20'),
    (45, 1, 'Tecnico', 'Estadistica aplicada', 4, 5400, '2026-06-04'),
    (46, 12, 'Novela', 'El olvido que seremos', 24, 1500, '2026-06-28'),
    (47, 15, 'Novela', 'El olvido que seremos', 39, 1500, '2026-01-15'),
    (48, 16, 'Tecnico', 'Redes', 32, 21000, '2026-04-17'),
]

cur = con.cursor()
cur.executemany("INSERT INTO lectores VALUES (%s, %s, %s, %s, %s)", lectores)
cur.executemany("INSERT INTO prestamos VALUES (%s, %s, %s, %s, %s, %s, %s)", prestamos)
con.commit()
print("Datos cargados:", len(lectores), "lectores y", len(prestamos), "prestamos")

## Consultas
Resuelva cada consulta en su celda. El resultado debe mostrarse como una tabla.

### Consulta 1. SELECT y TOP
Muestre los **primeros 8 lectores**, ordenados por `sede`.

In [ ]:
# Consulta 1. SELECT y TOP
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 2. Variables y filtro
Defina en Python una variable `dias_minimos = 30` e insértela con una f-string. Muestre los préstamos cuyo `dias_prestamo` es **mayor o igual** al mínimo, ordenados de mayor a menor por días.

In [ ]:
# Consulta 2. Variables y filtro
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 3. Limitar y paginar (TOP y OFFSET-FETCH)
Muestre los **4 préstamos con mayor multa** y, en otra tabla, los **4 siguientes** (paginación).

In [ ]:
# Consulta 3. Limitar y paginar (TOP y OFFSET-FETCH)
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 4. Funciones de fecha
Muestre la fecha actual, la fecha dentro de **15 días** y los días transcurridos desde el **1 de marzo de 2026**. En otra tabla, el número de préstamos y el **promedio de días** por mes.

In [ ]:
# Consulta 4. Funciones de fecha
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 5. Funciones de texto
Para cada lector, muestre el nombre, el nombre en **minúsculas**, su longitud y el **apellido** (la parte que va después del primer espacio).

In [ ]:
# Consulta 5. Funciones de texto
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 6. Conversión de tipos
Muestre la `multa` convertida a entero y la `fecha_prestamo` como texto en formato **dd.mm.aaaa** (estilo 104), para los 5 primeros préstamos. En otra tabla, demuestre `TRY_CONVERT` con '2024' y con 'x12'.

In [ ]:
# Consulta 6. Conversión de tipos
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 7. Manejo de nulos
Muestre el nombre, el `programa` original y el programa corregido, reemplazando los nulos por **'No registrado'** con `COALESCE`.

In [ ]:
# Consulta 7. Manejo de nulos
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 8. Lógica condicional (CASE e IIF)
Clasifique cada préstamo con `IIF` en 'Con multa' o 'Sin multa' según la multa sea mayor que cero, y con `CASE` la duración en 'Largo' (30 días o más), 'Medio' (15 a 29) o 'Corto' (menos de 15).

In [ ]:
# Consulta 8. Lógica condicional (CASE e IIF)
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 9. Expresiones de tabla comunes (CTE)
Con una CTE, calcule por categoría el **total de multa**, el número de préstamos y el **promedio de días**. Muestre solo las categorías cuyo total de multa supera **30000**.

In [ ]:
# Consulta 9. Expresiones de tabla comunes (CTE)
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Consulta 10. Funciones de ventana (OVER)
Con `ROW_NUMBER` y `PARTITION BY`, obtenga los **2 préstamos con mayor multa de cada categoría**.

In [ ]:
# Consulta 10. Funciones de ventana (OVER)
# Escriba aqui su consulta con pd.read_sql(...)
# Recuerde terminar la celda con df (o display(df)) para ver la tabla.



### Cierre
Al terminar, cierre la conexión con el motor.

In [ ]:
con.close()
print("Conexión cerrada")